
## Read Silver Data

#### Merge to delta tables

In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 7, Finished, Available, Finished, False)

In [11]:
%%sql
SELECT * from june_sales limit 5

StatementMeta(, 1341221c-6dce-40b9-a106-9214d1bd7f90, 13, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 13 fields>

In [10]:
df =spark.table("june_sales")
df_new= df.drop("iscount", "Currency")
df_new.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("June_Sales")

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 15, Finished, Available, Finished, False)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `june_sales` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [june_sales], [], false


In [15]:
%%sql
MERGE INTO june_sales AS t
USING july_sales AS s
ON t.SaleID = s.SaleID
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

StatementMeta(, 1341221c-6dce-40b9-a106-9214d1bd7f90, 17, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [16]:
%%sql
SELECT count(*) FROM june_sales

StatementMeta(, 1341221c-6dce-40b9-a106-9214d1bd7f90, 18, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [3]:
%%sql
SELECT * from sales limit 10

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 4, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 12 fields>

In [4]:
df1 = spark.read.table("sales")

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 6, Finished, Available, Finished, False)

In [15]:
df1=df1.drop("Currency")

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 21, Finished, Available, Finished, False)

**Gold table 1: Daily sales summary by region**

In [16]:
df_region_summary= df1.groupBy("Region", "SaleDate")\
                    .agg(
                            round(sum("Amount"), 2).alias("total_sales"),
                            round(avg("Amount"), 2).alias("avg_order_value"),
                            count("SaleID").alias("transaction_count"),
                            countDistinct("CustomerID").alias("unique_customer")
                            )

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 22, Finished, Available, Finished, False)

In [13]:
df_region_summary.show()

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 18, Finished, Available, Finished, False)

+--------------+----------+-----------+---------------+-----------------+---------------+
|        Region|  SaleDate|total_sales|avg_order_value|transaction_count|unique_customer|
+--------------+----------+-----------+---------------+-----------------+---------------+
|Southeast Asia|2024-06-04| 6229982.99|        7634.78|              816|            197|
|        Europe|2024-07-01| 6732387.09|        7254.73|              928|            197|
|   Middle East|2024-06-06| 6564270.96|        8074.13|              813|            197|
|        Europe|2024-06-02| 6288136.19|         7494.8|              839|            197|
|   Middle East|2024-07-03| 7261539.47|        7733.27|              939|            198|
|        Europe|2024-07-02| 7245385.07|        7892.58|              918|            197|
|        Europe|2024-06-01| 6321428.22|        7333.44|              862|            197|
|   Middle East|2024-06-03| 6566394.47|        7556.27|              869|            199|
|        E

**Gold table 2: Customer level summary**

In [24]:
df_customer_summary=df1.groupBy("CustomerID", "SaleDate")\
                        .agg(
                            round(sum("Amount"), 2).alias("total_spend"),
                            count("SaleID").alias("order_count"),
                            sum("Quantity").alias("total_items"),
                            round(avg("Amount"), 2).alias("avg_order_value")
                        )

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 30, Finished, Available, Finished, False)

In [19]:
df_region_summary.write\
                .format("delta")\
                .mode("overWrite")\
                .saveAsTable("gold_daily_region_sales")

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 25, Finished, Available, Finished, False)

In [25]:
df_customer_summary.write\
.format('delta')\
.mode("overWrite")\
.saveAsTable("gold_customer_daily_summary")

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 31, Finished, Available, Finished, False)

In [22]:
%%sql
SELECT * from gold_daily_region_sales 
ORDER by unique_customer desc limit 10

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 28, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 6 fields>

In [26]:
%%sql
SELECT * from gold_customer_daily_summary LIMIT 5

StatementMeta(, f5ad4839-10a0-47c2-b4d2-8c5c4ece2fd7, 32, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 6 fields>